In [1]:
import os 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma 
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

C:\Users\ACER\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
groq_api_key=os.getenv('GROQ_API_KEY')

model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000014A6A9EF010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000014A6AAA8B10>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
embeddings=HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6746.52it/s]


In [4]:
documents='''
# About Me — Kunal Chopde

## Professional Background
I am a Python Developer specializing in Generative AI, Agentic AI, and AI-powered application development. My primary goal is to build practical AI solutions that solve real-world problems, automate repetitive tasks, improve productivity, and create business value.

## Technical Skills
- **Programming:** Python, SQL
- **Generative AI:** LLMs, Prompt Engineering, Retrieval-Augmented Generation (RAG), embeddings, document question-answering systems
- **AI Agents:** LangChain, LangGraph, AutoGen, tool calling, multi-agent workflows, human-in-the-loop systems
- **Backend Development:** FastAPI, Flask, REST APIs
- **AI Application Interfaces:** Streamlit, Gradio, React/Vite
- **Machine Learning:** NumPy, Pandas, scikit-learn, statistics, model evaluation, data preprocessing
- **Databases and Search:** SQL Server, PostgreSQL, MySQL, SQLite, FAISS, Chroma
- **Development and Deployment:** Git, Docker, MLflow, API integration, environment configuration

## Projects and Practical Experience
I have worked on or developed projects involving:
- RAG-based document question-answering applications.
- AI research assistants using LangGraph, Groq, and web-search tools.
- AI agents that use tools to perform tasks and gather information.
- FastAPI-based Generative AI applications.
- Machine learning automation and MLOps project generators.
- AI business platforms for selling ready-made AI products and accepting custom development orders.
- Voice-enabled AI assistants and AI solutions for local businesses.

## Development Preferences
I prefer complete, runnable implementations with a clear folder structure, correct imports, dependency lists, environment configuration, and step-by-step setup instructions. I want code that works in my existing environment rather than unnecessary abstractions or overly complicated architecture.

I prefer practical examples, direct explanations, and minimal changes when debugging existing code. I primarily use Python and do not rely on Java for my development work.

I prefer affordable or free tools, including Groq and open-source models, wherever practical. I work within hardware and budget constraints and prioritize solutions that can be developed without expensive GPUs or paid subscriptions.

For AI applications, I value clean user interfaces, dark-themed designs, secure server-side API key handling, reliable outputs, source citations where appropriate, and straightforward deployment.

## Career Goals
My goal is to grow as a Generative AI and Agentic AI Developer, strengthen my portfolio, improve my resume and interview performance, and secure a paid entry-level or early-career AI development role.

I am interested in opportunities in India, particularly remote or hybrid roles involving Python, LLMs, RAG, AI agents, and backend development.

I also want to improve my practical engineering skills, understand modern AI frameworks, debug technical issues independently, and build production-oriented projects that demonstrate my capabilities.

## Entrepreneurship and Business Goals
I want to build and sell AI applications and agents to individuals and businesses that need automation but lack the technical expertise to develop these solutions themselves.

My interests include:
- AI assistants for local shops and small businesses.
- Customer-support and knowledge-base agents.
- Business process automation.
- Voice-enabled AI assistants.
- AI research and document analysis tools.
- A marketplace for ready-made AI products and custom AI development services.

I want to start with a zero or very limited budget, validate real customer problems, contact potential clients directly, and earn my first paying customer before investing heavily in a business.

## Education and Learning Style
I am pursuing postgraduate studies in computer applications while developing my technical skills and pursuing career opportunities.

I prefer concise, teacher-style explanations, practical demonstrations, important exam questions, structured study plans, and interview-oriented preparation. My learning plans should balance university studies, coding practice, job preparation, and business development.

## How ChatGPT Should Help Me
1. Give accurate, practical answers tailored to my Python and AI development background.
2. Provide complete runnable code when I request an implementation.
3. Keep solutions straightforward and avoid unnecessary features.
4. Explain errors clearly and fix the actual cause rather than suggesting unrelated changes.
5. Prefer free or low-cost tools and explain when a paid service is genuinely necessary.
6. Help me build distinctive AI products that solve validated customer problems rather than generic chatbot clones.
7. Support my career through relevant job searches, ATS-friendly resumes, interview preparation, and portfolio development.
8. Help me turn project ideas into working applications, deployable products, and potential sources of income.
9. Be honest about limitations, technical trade-offs, and what has or has not been verified.
10. Prioritize actionable next steps that fit around my studies, available time, and budget.

'''

In [5]:
from langchain_core.documents import Document

docs=[
    Document(
        page_content=documents
    )
]

In [6]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits=text_splitter.split_documents(docs)
vectorestore=Chroma.from_documents(documents=splits,embedding=embeddings)
retriever=vectorestore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000014A6AD0ECD0>, search_kwargs={})

In [7]:
system_prompt=(
    "you are an assistant for question-answering tasks."
    "use the following pieces of retrieval context to answer"
    "the question.If you don't know the answer, say that you"
    "don't know.Use three sentences maximum and keep the"
    "answer concise."
    "\n\n"
    "{context}"
)

prompt=ChatPromptTemplate.from_messages(
    [
        ('system',system_prompt),
        ('human','{input}')
    ]
)

In [8]:
answer_question_chain=create_stuff_documents_chain(model,prompt)
rag_chain=create_retrieval_chain(retriever,answer_question_chain)

In [9]:
response=rag_chain.invoke({'input':'What is AI'})
response

{'input': 'What is AI',
 'context': [Document(id='293279a3-73b8-48d5-809f-1d13249cf9ba', metadata={}, page_content='# About Me — Kunal Chopde\n\n## Professional Background\nI am a Python Developer specializing in Generative AI, Agentic AI, and AI-powered application development. My primary goal is to build practical AI solutions that solve real-world problems, automate repetitive tasks, improve productivity, and create business value.'),
  Document(id='509ee5c5-a595-4b40-b09a-9a3a817e2966', metadata={}, page_content='I am interested in opportunities in India, particularly remote or hybrid roles involving Python, LLMs, RAG, AI agents, and backend development.\n\nI also want to improve my practical engineering skills, understand modern AI frameworks, debug technical issues independently, and build production-oriented projects that demonstrate my capabilities.\n\n## Entrepreneurship and Business Goals\nI want to build and sell AI applications and agents to individuals and businesses that 

In [10]:
response['answer']

'Artificial intelligence (AI) is the field of computer science that creates systems capable of performing tasks that normally require human intelligence, such as reasoning, learning, perception, and problem‑solving. These systems use algorithms, models, and data to mimic cognitive functions and adapt to new information. AI spans subfields like machine learning, natural language processing, computer vision, and robotics, enabling applications from chatbots to autonomous vehicles.'

In [11]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt=(
    'Given a chat history and the latest user question'
    'which might reference context in the chat history,'
    'formualte a standalone question which can be understood'
    'without the chat history.Do NOT answer the question,'
    'just reformulate it if needed and otherwise return it as it is'
)

contextualize_system_prompt=ChatPromptTemplate.from_messages(
    [
        ('system',contextualize_q_system_prompt),
        MessagesPlaceholder('chat_history'),
        ('human','{input}')
    ]
)

In [12]:
history_aware_retriever=create_history_aware_retriever(model,retriever,contextualize_system_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000014A6AD0ECD0>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

In [13]:
qa_prompt=ChatPromptTemplate.from_messages(
    [
        ('system',system_prompt),
        MessagesPlaceholder('chat_history'),
        ('human',"{input}")
    ]
)

In [14]:
question_answer_chain=create_stuff_documents_chain(model,qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [15]:
from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
question='who is kunal chopde'
response1=rag_chain.invoke({'input':question,'chat_history':chat_history})
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1['answer'])
    ]

)
question='Tell me more about him'
response2=rag_chain.invoke({'input':question,'chat_history':chat_history})
print(response2['answer'])

Kunal Chopde is a Python developer who specializes in generative AI, agentic AI, and AI‑powered application development, aiming to build practical, business‑value solutions. He is proficient in Python, SQL, LLMs, LangChain, LangGraph, AutoGen, FastAPI, Streamlit, and open‑source models like Groq, focusing on clean interfaces and secure API key handling. His career goal is to grow as an AI developer, strengthen his portfolio, and secure a paid entry‑level or early‑career AI role, especially in India’s remote or hybrid settings.
